# Generate Test Data
Generates data in tests/test_data/




In [ ]:

#import sys
#sys.path.insert(0, "/mnt/c/Users/hans/OneDrive/Python3/packages/cdxcore") # do this before importing anything else

import os
os.environ["OQ_DATA_TEST_MODE"] = "generate_tests" # do this before importing anything else

%load_ext autoreload
%autoreload 1

%aimport openqdata
%aimport openqdata.biz_calendar
%aimport openqdata.arbfree.arbfree
%aimport openqdata.widb
%aimport openqdata.widb.client
%aimport openqdata.widb.day
%aimport openqdata.widb.arbfree

from openqdata import env# triggers construction of caches.
from openqdata.biz_calendar import next_business_day
from cdxcore.verbose import Context

def generate_day(ticker, d):
    from openqdata.widb.day import collect_raw_day, enrich_day, ExpiryFilter, RawDayFilter
    from openqdata.widb.arbfree import mk_arb_free_day, MkArbFree

    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    mk_arb_free = MkArbFree(timeout_seconds=60*20)
    verbose = Context.all

    raw_day = collect_raw_day(ticker, d, raw_day_filter=raw_day_filter, verbose=verbose, raise_if_bad_day=False)
    enr_day = enrich_day(ticker, d, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad_day=False, override_cache_mode="off")

    buggy   = enr_day[enr_day['pure_ask_price'] <= enr_day['pure_bid_price']]
    assert buggy.empty, f"Found {len(buggy)} rows with ask price <= bid price for {ticker} {d}:\n{buggy[['expiry','strike','spot','pure_ask_price', 'pure_bid_price','call_ask','call_bid','mkt_call_ask', 'mkt_call_bid','mkt_put_ask', 'mkt_put_bid']]}"


    af_day = mk_arb_free_day(ticker, d, mk_arb_free=mk_arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, raise_if_bad_day=False)

tickers = ["SPX", "AAPL"]
dates = ["2006-01-31", "2005-11-18", "2024-03-03", "2025-01-02", "2024-12-24", "2010-07-05", "2011-07-06"]
dates = [next_business_day(d,count_start=True) for d in dates]

for ticker in tickers:
    for d in dates:
        print(f"Generating data for {ticker} {d}")
        generate_day(ticker, d)
